# Lesson 1b — HJ-WENO5 for level-set advection

Prereq: Lesson 1. Reference: Jiang & Peng (2000); Osher & Fedkiw, *Level Set Methods and Dynamic Implicit Surfaces*, Ch. 3.

Goals:
1. Understand why first-order upwind advection (Lesson 1) isn't good enough for a real fire solver.
2. Derive the HJ-WENO5 scheme: five candidate differences, three 3rd-order stencils, nonlinear weights.
3. Implement it in vectorized NumPy.
4. Compare upwind-1 vs HJ-WENO5 on a clean test: rigid rotation of a disc. HJ-WENO5 preserves the shape; upwind-1 smears it away.

## 1. Why not just upwind-1?

Upwind-1 (first-order upwind finite difference) has truncation error `O(h)` with an implicit numerical viscosity proportional to `|w|·h`. For level sets this viscosity **shrinks and smears the zero set** — exactly the thing we don't want:

- A fuel disc under pure rotation should stay a circle. With upwind-1 it blurs into a disc of lower-magnitude `φ` values, the zero-contour shrinks, and you need aggressive reinitialization to recover anything resembling a circle. After many steps the disc is just gone.
- The Darrieus–Landau instability (Paper 3) is linear and proportional to wavenumber. Upwind-1's diffusion dominates the growth of any resolvable wavelength — so wrinkles never develop even with `M < 0`.

HJ-WENO5 is the standard fix. It is 5th-order accurate in smooth regions and non-oscillatory across kinks (the SDF skeleton, cusps from reinit, corners of the zero set).

## 2. The setup: Hamilton–Jacobi form

Level-set advection is a Hamilton–Jacobi equation:

$$\frac{\partial \phi}{\partial t} + H(\nabla \phi) = 0, \qquad H(\nabla \phi) = \mathbf{w} \cdot \nabla \phi$$

In 1-D with velocity `u`: `∂φ/∂t + u·φ_x = 0`. The only numerical choice is how to approximate `φ_x` at each grid point. Upwind-1 uses a two-point difference; HJ-WENO5 uses a clever weighted combination of three 3rd-order stencils built from five local first differences.

### Why separate `φ_x⁻` and `φ_x⁺`?

For stability, we need an **upwind** derivative: look in the direction information comes *from*. In 1-D:
- If `u > 0`, info comes from the left → use `φ_x⁻` (built from stencils biased left).
- If `u < 0`, info comes from the right → use `φ_x⁺` (biased right).

Then `φ_x_used = (u > 0) ? φ_x⁻ : φ_x⁺`. Same idea per dimension in 2-D and 3-D.

## 3. Constructing `φ_x⁻`

At grid point `i`, compute five consecutive first differences (all divided by `h`):

```
v1 = (φ_{i-2} − φ_{i-3}) / h
v2 = (φ_{i-1} − φ_{i-2}) / h
v3 = (φ_i     − φ_{i-1}) / h
v4 = (φ_{i+1} − φ_i    ) / h
v5 = (φ_{i+2} − φ_{i+1}) / h
```

These are the slopes on successive edges of a six-cell stencil `{i-3, i-2, i-1, i, i+1, i+2}`.

### Three candidate 3rd-order approximations

Each sub-stencil of four consecutive points gives a 3rd-order approximation to `φ_x(x_i)`:

$$
\phi_x^{(1)} = \tfrac{1}{3}v_1 - \tfrac{7}{6}v_2 + \tfrac{11}{6}v_3 \\
\phi_x^{(2)} = -\tfrac{1}{6}v_2 + \tfrac{5}{6}v_3 + \tfrac{1}{3}v_4 \\
\phi_x^{(3)} = \tfrac{1}{3}v_3 + \tfrac{5}{6}v_4 - \tfrac{1}{6}v_5
$$

The *linear* combination with ideal weights `d = (1/10, 6/10, 3/10)` is 5th-order accurate in smooth regions:

$$
\phi_x^{\text{ideal}} = \tfrac{1}{10}\phi_x^{(1)} + \tfrac{6}{10}\phi_x^{(2)} + \tfrac{3}{10}\phi_x^{(3)}
$$

But those fixed weights cause oscillations when any of the three sub-stencils straddles a kink. WENO replaces `d_k` with *nonlinear* weights that shrink the contribution of any rough stencil.

## 4. Smoothness indicators and nonlinear weights

Jiang–Shu smoothness indicators measure how rough each sub-stencil is:

$$
S_1 = \tfrac{13}{12}(v_1 - 2v_2 + v_3)^2 + \tfrac{1}{4}(v_1 - 4v_2 + 3v_3)^2 \\
S_2 = \tfrac{13}{12}(v_2 - 2v_3 + v_4)^2 + \tfrac{1}{4}(v_2 - v_4)^2 \\
S_3 = \tfrac{13}{12}(v_3 - 2v_4 + v_5)^2 + \tfrac{1}{4}(3v_3 - 4v_4 + v_5)^2
$$

Each is `O(h²)` in smooth regions and `O(1)` where the sub-stencil crosses a kink — the key property that drives adaptivity.

Nonlinear weights:

$$
\alpha_k = \frac{d_k}{(\epsilon + S_k)^2}, \qquad w_k = \frac{\alpha_k}{\alpha_1 + \alpha_2 + \alpha_3}
$$

with `ε ≈ 10⁻⁶` to avoid division by zero. In smooth regions `S_k ≈ O(h²)` so the `w_k` converge to `d_k` and we recover 5th-order accuracy. Near a kink, the rough sub-stencil has a much larger `S_k`, its `w_k` → 0, and the scheme "picks" the smooth neighbors — essentially non-oscillatory behavior.

Final:

$$\phi_x^- = w_1 \phi_x^{(1)} + w_2 \phi_x^{(2)} + w_3 \phi_x^{(3)}$$

### And `φ_x⁺`?

Mirror the stencil: redefine

```
v1 = (φ_{i+3} − φ_{i+2}) / h
v2 = (φ_{i+2} − φ_{i+1}) / h
v3 = (φ_{i+1} − φ_i    ) / h
v4 = (φ_i     − φ_{i-1}) / h
v5 = (φ_{i-1} − φ_{i-2}) / h
```

and apply the same formulas. Same code, different data stride.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def weno5_core(v1, v2, v3, v4, v5, eps=1e-6):
    """Core 5th-order WENO reconstruction from five differences.
    Input arrays can be any shape; works element-wise."""
    # Three candidate 3rd-order approximations
    p1 = v1/3 - 7*v2/6 + 11*v3/6
    p2 = -v2/6 + 5*v3/6 + v4/3
    p3 = v3/3 + 5*v4/6 - v5/6
    # Smoothness indicators (Jiang-Shu)
    S1 = 13/12 * (v1 - 2*v2 + v3)**2 + 1/4 * (v1 - 4*v2 + 3*v3)**2
    S2 = 13/12 * (v2 - 2*v3 + v4)**2 + 1/4 * (v2 - v4)**2
    S3 = 13/12 * (v3 - 2*v4 + v5)**2 + 1/4 * (3*v3 - 4*v4 + v5)**2
    # Nonlinear weights (ideal: 0.1, 0.6, 0.3)
    a1 = 0.1 / (eps + S1)**2
    a2 = 0.6 / (eps + S2)**2
    a3 = 0.3 / (eps + S3)**2
    asum = a1 + a2 + a3
    return (a1*p1 + a2*p2 + a3*p3) / asum

## 5. Vectorized 2-D implementation

Apply WENO5 along one axis at a time. Use `np.pad(mode='edge')` for Neumann BCs (same as in Lesson 1).

In [ ]:
def hj_weno5_derivs(phi, h):
    """Return (phi_x_minus, phi_x_plus, phi_y_minus, phi_y_plus), all same shape as phi."""
    # Pad by 3 in each direction with edge-replication for Neumann BCs.
    p = np.pad(phi, 3, mode='edge')
    Ny, Nx = phi.shape

    def slab(axis, offset):
        """Return the interior slice shifted by `offset` along `axis`, still same shape as phi."""
        s = [slice(3, 3+Ny), slice(3, 3+Nx)]
        s[axis] = slice(3+offset, 3+offset + phi.shape[axis])
        return p[tuple(s)]

    # x-direction first differences: dx_k = (phi at offset k) - (phi at offset k-1), /h
    # φ_x⁻ uses stencil {i-3..i+2}, so v1..v5 are differences at offsets (-2,-1,0,+1,+2)
    def dplus_minus(axis):
        # Build v1..v5 for φ_(axis)⁻
        vm1 = (slab(axis, -2) - slab(axis, -3)) / h
        vm2 = (slab(axis, -1) - slab(axis, -2)) / h
        vm3 = (slab(axis,  0) - slab(axis, -1)) / h
        vm4 = (slab(axis,  1) - slab(axis,  0)) / h
        vm5 = (slab(axis,  2) - slab(axis,  1)) / h
        phix_minus = weno5_core(vm1, vm2, vm3, vm4, vm5)
        # Mirror for φ_(axis)⁺: use differences at offsets (+3,+2,+1,0,-1)
        vp1 = (slab(axis,  3) - slab(axis,  2)) / h
        vp2 = (slab(axis,  2) - slab(axis,  1)) / h
        vp3 = (slab(axis,  1) - slab(axis,  0)) / h
        vp4 = (slab(axis,  0) - slab(axis, -1)) / h
        vp5 = (slab(axis, -1) - slab(axis, -2)) / h
        phix_plus = weno5_core(vp1, vp2, vp3, vp4, vp5)
        return phix_minus, phix_plus

    phi_y_m, phi_y_p = dplus_minus(axis=0)
    phi_x_m, phi_x_p = dplus_minus(axis=1)
    return phi_x_m, phi_x_p, phi_y_m, phi_y_p


def hj_weno5_advect(phi, u, v, dt, h):
    """One forward-Euler HJ-WENO5 advection step for ∂φ/∂t + u φ_x + v φ_y = 0.
    For production use, wrap this in a TVD Runge-Kutta 3 time integrator."""
    fxm, fxp, fym, fyp = hj_weno5_derivs(phi, h)
    phi_x = np.where(u > 0, fxm, fxp)
    phi_y = np.where(v > 0, fym, fyp)
    return phi - dt * (u * phi_x + v * phi_y)


def hj_weno5_advect_rk3(phi, u, v, dt, h):
    """TVD RK3 (Shu-Osher) + HJ-WENO5 spatial — the production-grade combo."""
    phi1 = hj_weno5_advect(phi, u, v, dt, h)
    phi2 = 0.75 * phi + 0.25 * hj_weno5_advect(phi1, u, v, dt, h)
    return (1/3) * phi + (2/3) * hj_weno5_advect(phi2, u, v, dt, h)

## 6. Visual comparison — rigid rotation of a disc

Classic test: rotate a disc around the domain center. The exact solution at any time is the same disc, rotated. Any numerical scheme's error shows up as shape distortion. We time the simulations, compare upwind-1 from Lesson 1 against HJ-WENO5 + RK3 here.

In [ ]:
N = 128
L = 2.0
h = L / N
x = np.linspace(-L/2, L/2, N); y = x.copy()
X, Y = np.meshgrid(x, y, indexing='xy')

# Off-center disc so rotation is visible
cx, cy, R = 0.4, 0.0, 0.25
phi0 = np.sqrt((X - cx)**2 + (Y - cy)**2) - R

# Rigid rotation velocity: omega = 1 rad/s, so full turn at t = 2π
omega = 1.0
U = -omega * Y
V =  omega * X

# Upwind-1 (from Lesson 1, with Neumann BCs)
def shift_y(phi, d):
    p = np.pad(phi, ((1,1),(0,0)), mode='edge'); return p[1-d:1-d+phi.shape[0], :]
def shift_x(phi, d):
    p = np.pad(phi, ((0,0),(1,1)), mode='edge'); return p[:, 1-d:1-d+phi.shape[1]]
def upwind1(phi, u, v, dt, h):
    pxp = (shift_x(phi,-1) - phi) / h
    pxm = (phi - shift_x(phi,+1)) / h
    pyp = (shift_y(phi,-1) - phi) / h
    pym = (phi - shift_y(phi,+1)) / h
    return phi - dt * (u * np.where(u>0, pxm, pxp) + v * np.where(v>0, pym, pyp))

vmax = max(np.abs(U).max(), np.abs(V).max())
dt = 0.4 * h / vmax
T_final = 2 * np.pi      # one full rotation
nsteps = int(np.ceil(T_final / dt))
dt = T_final / nsteps

phi_u1 = phi0.copy()
phi_w5 = phi0.copy()
for _ in range(nsteps):
    phi_u1 = upwind1(phi_u1, U, V, dt, h)
    phi_w5 = hj_weno5_advect_rk3(phi_w5, U, V, dt, h)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, p, title in zip(
    ax,
    [phi0, phi_u1, phi_w5],
    ['initial (= exact after one turn)', 'upwind-1 after one full rotation', 'HJ-WENO5 + RK3 after one full rotation']
):
    a.contourf(X, Y, p, levels=np.linspace(-0.3, 0.3, 21), cmap='RdBu')
    a.contour(X, Y, p, levels=[0], colors='k', linewidths=2)
    a.set_aspect('equal'); a.set_title(title)
plt.tight_layout(); plt.show()

# Quantify: area of zero level set should be π R² = 0.196
def inside_area(phi, h):
    return (phi < 0).sum() * h * h
A_exact = np.pi * R**2
print(f'Expected area: {A_exact:.4f}')
print(f'Upwind-1 area after one turn:      {inside_area(phi_u1, h):.4f}')
print(f'HJ-WENO5+RK3 area after one turn:  {inside_area(phi_w5, h):.4f}')

You should see the upwind-1 disc dramatically shrunken and smeared (sometimes gone entirely depending on grid), while the HJ-WENO5+RK3 disc is visually indistinguishable from the initial condition. That's the practical payoff.

## 7. Cost

- Per-cell stencil is **6 cells wide in each direction**, vs. 3 for upwind-1. Memory traffic is ~2x.
- Arithmetic per cell is ~20x upwind-1 (three sub-stencils, three smoothness indicators, weighted combo, done for both `⁻` and `⁺`).
- But the level set only needs to be advected in a **narrow band** around the front (width `~5h`), not the whole domain. In a production fire solver this makes HJ-WENO5 on the narrow band cheaper than running upwind-1 everywhere plus frequent reinitialization to correct the error.

## 8. Practical notes

- **Always pair HJ-WENO5 with TVD-RK3** (Shu–Osher) time integration. Forward Euler + WENO5 has a very tight CFL and can still oscillate at kinks.
- CFL: `dt ≤ 0.4 · h / max|w|` is the safe default for RK3+WENO5.
- `ε = 10⁻⁶` is the standard small-number in the smoothness denominator. Too small and you get division-by-zero cascades in flat regions; too large and you lose WENO's adaptivity.
- Reinitialization (Lesson 1, §6) should also use HJ-WENO5 differences if you want to preserve sharp features on the front.
- The same WENO5 reconstruction is used in the *conservative* form for scalar transport (temperature, soot) via a flux-split formulation; different packaging, same core idea.

## Exercises

1. Replace RK3 with forward Euler in the rotation test. Observe that `dt` has to shrink and the disc still looks good, but more steps are needed.
2. Rotate a **square** (sharp corners) instead of a disc. Upwind-1 rounds the corners; HJ-WENO5 preserves them much longer, though you'll still see tiny oscillations — this is where the WENO adaptivity earns its keep.
3. Add a flame speed `S` to the rotation (as in Lesson 1, Exercise 3). Run with HJ-WENO5 and confirm the disc both rotates and shrinks correctly.
4. In Lesson 1, swap the reinit scheme to use WENO5 differences instead of the simple upwind Godunov selection. Compare on a non-SDF initial condition.